In [1]:
from math import gcd

def chinese_remainder(n, a):
    """Standard CRT solver to combine results."""
    sum = 0
    prod = 1
    for n_i in n: prod *= n_i
    for n_i, a_i in zip(n, a):
        p = prod // n_i
        sum += a_i * pow(p, -1, n_i) * p
    return sum % prod

def pohlig_hellman(g, h, p, factors):
    """
    Solves g^x = h (mod p) given prime factors of p-1.
    'factors' is a list of (q, e) where q^e is a factor.
    """
    n = p - 1
    results = []
    moduli = []

    for q, e in factors:
        # Solve for x mod q^e
        q_e = q**e
        # In this simplified demo, we use brute force for the subgroup
        # In a real tool, you'd use Baby-step Giant-step here.
        sub_g = pow(g, n // q_e, p)
        sub_h = pow(h, n // q_e, p)
        
        found = False
        for x_i in range(q_e):
            if pow(sub_g, x_i, p) == sub_h:
                results.append(x_i)
                moduli.append(q_e)
                found = True
                break
        if not found:
            raise Exception(f"Failed to find log in subgroup {q}^{e}")

    # Combine results using CRT
    return chinese_remainder(moduli, results)

# --- DEMO CASE ---
# Solve 3^x = 37 (mod 101)
# p-1 = 100 = 2^2 * 5^2
p = 101
g = 3
h = 37
factors = [(2, 2), (5, 2)] # 4 and 25

secret_x = pohlig_hellman(g, h, p, factors)
print(f"The discrete log x is: {secret_x}")
print(f"Verification: {g}^{secret_x} % {p} = {pow(g, secret_x, p)}")

The discrete log x is: 24
Verification: 3^24 % 101 = 37
